In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class SimpleGraphSAGE(nn.Module):
    def __init__(self,input_dim,hidden_dim):
        super(SimpleGraphSAGE,self).__init__()
        self.linear_self=nn.Linear(input_dim,hidden_dim)
        self.linear_neigh=nn.Linear(input_dim,hidden_dim)
    def forward(self,x,adj):
        neighbor_sum=torch.matmul(adj,x)
        degree=adj.sum(dim=1,keepdim=True)+1e-6
        neighbor_mean=neighbor_sum/degree
        self_features=self.linear_self(x)
        neigh_features=self.linear_neigh(neighbor_mean)
        out=self_features+neigh_features
        return F.relu(out)
class MLPClassifier(nn.Module):
    def __init__(self,input_dim):
        super(MLPClassifier,self).__init__()
        self.fc1=nn.Linear(input_dim,64)
        self.fc2=nn.Linear(64,32)
        self.fc3=nn.Linear(32,2)
    def forward(self,x):
        x=F.relu(self.fc1(x))
        x=F.dropout(x,p=0.5,training=self.training)
        x=F.relu(self.fc2(x))
        x=F.dropout(x,p=0.5,training=self.training)
        x=self.fc3(x)
        return F.softmax(x,dim=1)
class MLPRanker(nn.Module):
    def __init__(self,input_dim):
        super(MLPRanker,self).__init__()
        self.fc1=nn.Linear(input_dim,64)
        self.fc2=nn.Linear(64,32)
        self.fc3=nn.Linear(32,1)
    def forward(self,x):
        x=F.relu(self.fc1(x))
        x=F.dropout(x,p=0.5,training=self.training)
        x=F.relu(self.fc2(x))
        x=F.dropout(x,p=0.5,training=self.training)
        x=self.fc3(x)
        return x
class BetweennessModel(nn.Module):
    def __init__(self,node_feat_dim,hidden_dim):
        super(BetweennessModel,self).__init__()
        self.gnn1=SimpleGraphSAGE(node_feat_dim,hidden_dim)
        self.gnn2=SimpleGraphSAGE(hidden_dim,hidden_dim)
        self.classifier=MLPClassifier(hidden_dim*2)
        self.ranker=MLPRanker(hidden_dim*2)
    def forward(self,x,adj,edge_pairs):
        h=self.gnn1(x,adj)
        h=self.gnn2(h,adj)
        u=edge_pairs[:,0]
        v=edge_pairs[:,1]
        edge_embeddings=torch.cat( [h[u],h[v]],dim=1)
        class_probs=self.classifier(edge_embeddings)
        rank_scores=self.ranker(edge_embeddings)
        return class_probs,rank_scores
def classification_loss(pred_probs,labels):
    return F.binary_cross_entropy(pred_probs[:,1], labels.float())
def triplet_margin_loss(scores,triplets,margin=1.0):
    loss=0
    for t in triplets:
        m,n,o=t
        sm=scores[m]
        sn=scores[n]
        so=scores[o]
        loss1=torch.maximum(torch.tensor(0.0), -(sm-sn)+margin)
        loss2=torch.maximum(torch.tensor(0.0),-(sn-so)+margin)
        loss3=torch.maximum(torch.tensor(0.0),-(so-sm)+margin)
        loss+=loss1+loss2+loss3
    return loss/len(triplets)
num_nodes=100
feature_dim=16
x=torch.rand((num_nodes,feature_dim))
adj=torch.randint(0,2,(num_nodes,num_nodes)).float()
adj.fill_diagonal_(0)
candidate_edges=torch.randint(0,num_nodes,(500,2))
labels=torch.randint(0,2,(500,))
triplets=torch.randint(0,500,(100,3))
model=BetweennessModel(node_feat_dim=16,hidden_dim=32)
optimizer=torch.optim.Adam(model.parameters(),lr=0.0085)
model.train()
for epoch in range(50):
    optimizer.zero_grad()
    class_probs,rank_scores=model(x,adj,candidate_edges)
    loss_cls=classification_loss(class_probs,labels)
    loss_rank=triplet_margin_loss(rank_scores.squeeze(),triplets)
    total_loss=loss_cls+loss_rank
    total_loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {total_loss.item():.4f}")

Epoch 1, Loss: 3.6970
Epoch 2, Loss: 3.6938
Epoch 3, Loss: 3.6944
Epoch 4, Loss: 3.6927
Epoch 5, Loss: 3.6969
Epoch 6, Loss: 3.6923
Epoch 7, Loss: 3.6966
Epoch 8, Loss: 3.6928
Epoch 9, Loss: 3.6908
Epoch 10, Loss: 3.6919
Epoch 11, Loss: 3.6931
Epoch 12, Loss: 3.6904
Epoch 13, Loss: 3.6926
Epoch 14, Loss: 3.6896
Epoch 15, Loss: 3.6907
Epoch 16, Loss: 3.6924
Epoch 17, Loss: 3.6888
Epoch 18, Loss: 3.6859
Epoch 19, Loss: 3.6842
Epoch 20, Loss: 3.6921
Epoch 21, Loss: 3.6863
Epoch 22, Loss: 3.6844
Epoch 23, Loss: 3.6761
Epoch 24, Loss: 3.6785
Epoch 25, Loss: 3.6775
Epoch 26, Loss: 3.6680
Epoch 27, Loss: 3.6775
Epoch 28, Loss: 3.6786
Epoch 29, Loss: 3.6764
Epoch 30, Loss: 3.6711
Epoch 31, Loss: 3.6692
Epoch 32, Loss: 3.6691
Epoch 33, Loss: 3.6664
Epoch 34, Loss: 3.6707
Epoch 35, Loss: 3.6664
Epoch 36, Loss: 3.6612
Epoch 37, Loss: 3.6626
Epoch 38, Loss: 3.6598
Epoch 39, Loss: 3.6634
Epoch 40, Loss: 3.6556
Epoch 41, Loss: 3.6564
Epoch 42, Loss: 3.6645
Epoch 43, Loss: 3.6555
Epoch 44, Loss: 3.66